In [1]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/NTN_MEC_Ablation'
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Saving all checkpoints and results to: {BASE_DIR}")

Mounted at /content/drive
Saving all checkpoints and results to: /content/drive/MyDrive/NTN_MEC_Ablation


In [2]:
%%writefile agent.py

"""
Independent Learner (IL) agent — Dueling DQN.

Faithful to the paper:
  - Dueling architecture: V(s) + A(s,a) − mean(A)
  - Target network, hard-updated every C steps
  - Experience replay buffer
  - ε-greedy exploration
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random
from typing import Tuple, Optional


# ─── Replay Buffer ─────────────────────────────────────────────────────────

class ReplayBuffer:
    def __init__(self, capacity: int = 10_000):
        self.buf = deque(maxlen=capacity)

    def push(self, obs, action, reward, next_obs, done):
        self.buf.append((
            np.array(obs,      dtype=np.float32),
            int(action),
            float(reward),
            np.array(next_obs, dtype=np.float32),
            float(done),
        ))

    def sample(self, batch_size: int):
        batch = random.sample(self.buf, batch_size)
        obs, actions, rewards, next_obs, dones = zip(*batch)
        return (
            torch.tensor(np.stack(obs),      dtype=torch.float32),
            torch.tensor(actions,            dtype=torch.long),
            torch.tensor(rewards,            dtype=torch.float32),
            torch.tensor(np.stack(next_obs), dtype=torch.float32),
            torch.tensor(dones,              dtype=torch.float32),
        )

    def __len__(self):
        return len(self.buf)


# ─── Dueling DQN Network ───────────────────────────────────────────────────

class DuelingDQN(nn.Module):
    """
    Shared trunk → split into Value stream V(s) and Advantage stream A(s,a).
    Q(s,a) = V(s) + A(s,a) − mean_a′ A(s,a′)
    """

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 128):
        super().__init__()

        # Shared feature extractor
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )

        # Value stream
        self.value_stream = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

        # Advantage stream
        self.adv_stream = nn.Sequential(
            nn.Linear(hidden, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.trunk(x)
        V    = self.value_stream(feat)                      # (B, 1)
        A    = self.adv_stream(feat)                        # (B, n_actions)
        Q    = V + A - A.mean(dim=1, keepdim=True)         # (B, n_actions)
        return Q


# ─── IL Agent ──────────────────────────────────────────────────────────────

class ILAgent:
    """
    One independent dueling-DQN agent per UE.
    Trains from its own local replay buffer.
    """

    def __init__(
        self,
        obs_dim:      int,
        n_actions:    int,
        hidden:       int   = 128,
        lr:           float = 1e-3,       # paper: 0.001
        gamma:        float = 0.9,        # paper: 0.9
        buffer_cap:   int   = 10_000,
        batch_size:   int   = 32,         # paper: 32
        target_update:int   = 20,         # hard-copy every C steps
        eps_start:    float = 1.0,
        eps_end:      float = 0.05,
        eps_decay:    float = 0.995,
        device:       str   = "cpu",
    ):
        self.n_actions     = n_actions
        self.gamma         = gamma
        self.batch_size    = batch_size
        self.target_update = target_update
        self.device        = torch.device(device)

        # ε-greedy schedule
        self.eps       = eps_start
        self.eps_end   = eps_end
        self.eps_decay = eps_decay

        # Networks
        self.policy_net = DuelingDQN(obs_dim, n_actions, hidden).to(self.device)
        self.target_net = DuelingDQN(obs_dim, n_actions, hidden).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        # Optimiser (paper uses RMSProp)
        self.optimiser = torch.optim.RMSprop(self.policy_net.parameters(), lr=lr)

        self.buffer    = ReplayBuffer(buffer_cap)
        self.steps     = 0              # total training steps taken

    # ── Action selection ───────────────────────────────────────────────────

    def select_action(self, obs: np.ndarray, greedy: bool = False) -> int:
        if not greedy and random.random() < self.eps:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            q = self.policy_net(
                torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(self.device)
            )
            return int(q.argmax(dim=1).item())

    # ── Store transition ───────────────────────────────────────────────────

    def store(self, obs, action, reward, next_obs, done):
        self.buffer.push(obs, action, reward, next_obs, done)

    # ── Training step ──────────────────────────────────────────────────────

    def train_step(self) -> Optional[float]:
        """
        Sample one mini-batch and do one gradient update.
        Returns loss value (float) or None if buffer too small.
        """
        if len(self.buffer) < self.batch_size:
            return None

        obs, actions, rewards, next_obs, dones = [
            t.to(self.device) for t in self.buffer.sample(self.batch_size)
        ]

        # Current Q values
        q_values = self.policy_net(obs)                     # (B, n_actions)
        q_pred   = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)  # (B,)

        # Target Q values — Double DQN style:
        #   action chosen by policy_net, evaluated by target_net
        with torch.no_grad():
            next_actions = self.policy_net(next_obs).argmax(dim=1)       # (B,)
            next_q       = self.target_net(next_obs).gather(
                               1, next_actions.unsqueeze(1)).squeeze(1)  # (B,)
            q_target = rewards + self.gamma * next_q * (1.0 - dones)

        loss = F.smooth_l1_loss(q_pred, q_target)

        self.optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimiser.step()

        self.steps += 1

        # Hard target update every C steps
        if self.steps % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return float(loss.item())

    def decay_epsilon(self):
        """Call once per episode externally — not inside train_step."""
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

    # ── CTDE weight sync (used by CTDETrainer, not IL) ─────────────────────

    def set_weights(self, state_dict):
        self.policy_net.load_state_dict(state_dict)

    def get_weights(self):
        return self.policy_net.state_dict()

Writing agent.py


In [3]:
%%writefile env.py

"""
NTN-MEC Environment — faithful simplification of:
  Fatima, Saxena, Giambene. "Computation Offloading in NTN-empowered MEC
  using Multi-Agent Distributed Deep Reinforcement Learning." GLOBECOM 2024.

Simplifications vs. paper:
  - No LEO satellite (UAV-only for speed; easy to add back)
  - Smaller default scale: M=10 UEs, N=2 UAVs (paper: 40/4)
  - Same queue/delay/energy/reward math as the paper (Eqs 1-14)
"""

import numpy as np
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict


# ─── Configuration ─────────────────────────────────────────────────────────

@dataclass
class EnvConfig:
    # Topology
    M: int   = 10       # number of UEs
    N: int   = 2        # number of UAVs

    # Area & altitude
    L: float = 100.0    # m (area length)
    W: float = 100.0    # m (area width)
    H: float = 100.0    # m (UAV hovering altitude)

    # Time
    delta: float = 0.2  # time slot duration (s)
    I: int       = 100  # time slots per episode

    # Task model
    P_task: float = 0.3    # task generation probability per slot
    D_min:  float = 1.0    # min task size (Mbits)
    D_max:  float = 3.0    # max task size (Mbits)  [paper: 2–5, scaled down]
    sm:     float = 297.0  # CPU cycles per bit
    # Paper quotes 0.297 gigacycles/Mbit; since D_bits = D_mbits * 1e6,
    # the correct per-bit value is 0.297e9 / 1e6 = 297 cycles/bit.
    tau:    int   = 10     # deadline (time slots)

    # Communication (Eq. 1)
    B:       float = 1e6    # bandwidth per UE-UAV link (Hz) — 1 MHz
    Pup_uav: float = 0.1    # uplink power to UAV (W)
    beta0:   float = 1e-3   # path loss constant
    j:       float = 2.0    # path loss exponent
    sigma2:  float = 1e-10  # noise power (W) ≈ -100 dBm

    # Computation
    f_device:   float = 1e9   # UE CPU frequency (cycles/s) — 1 GHz
    f_edge_uav: float = 2.5e9 # UAV edge server frequency (cycles/s)

    # Energy (Eqs. 12, 13)
    kappa: float = 1e-27  # effective switched capacitance

    # Cost weights (Eq. 14)
    w1: float = 0.5   # delay weight
    w2: float = 0.5   # energy weight

    # Penalty for a dropped task
    penalty: float = 20.0

    # UE mobility
    v:    float = 1.0  # speed (m/s)
    tmov: float = 1.0  # movement time per slot (s)


# ─── Environment ───────────────────────────────────────────────────────────

class NTNMECEnv:
    """
    Multi-agent NTN-MEC offloading environment.

    Action space per agent:  {0=local, 1=UAV_0, ..., N=UAV_{N-1}}
    Observation per agent:
        [ue_pos(2) | uav_pos(2N) | task_size(1) | local_wait(1) |
         tran_wait(1) | edge_queue_per_uav(N) | active_queues_per_uav(N)]
    Total obs dim = 5 + 4*N  (13 for N=2)
    """

    def __init__(self, config: Optional[EnvConfig] = None):
        self.cfg = config or EnvConfig()
        cfg = self.cfg

        self.n_agents  = cfg.M
        self.n_actions = cfg.N + 1          # local + N UAVs
        # obs: ue_pos(2) + uav_pos(2N) + task(1) + local_wait(1) + tran_wait(1) + edge_q(N)
        # = 5 + 3*N   (active_q removed — GNN provides that signal instead)
        self.obs_dim   = 5 + 3 * cfg.N

        # Fixed UAV horizontal positions (evenly spaced along x-axis, y=W/2)
        self.uav_pos = np.array([
            [(n + 1) * cfg.L / (cfg.N + 1), cfg.W / 2]
            for n in range(cfg.N)
        ], dtype=np.float32)               # shape (N, 2) — horizontal only

        self.reset()

    # ── Reset ──────────────────────────────────────────────────────────────

    def reset(self) -> List[np.ndarray]:
        cfg = self.cfg
        self.t = 0

        # UE positions: uniform random in area
        self.ue_pos = np.random.uniform(
            [0, 0], [cfg.L, cfg.W], size=(cfg.M, 2)
        ).astype(np.float32)

        # UE movement angles
        self.ue_angles = np.random.uniform(0, 2 * np.pi, cfg.M)

        # Queue state: the time slot at which each queue becomes free.
        #   l_comp[m]    — local computation queue for UE m
        #   l_tran[m]    — transmission queue for UE m
        #   l_edge[n, m] — edge queue at UAV n for UE m
        # Initialised to 0 (free from slot 0 onwards).
        self.l_comp = np.zeros(cfg.M, dtype=np.float32)
        self.l_tran = np.zeros(cfg.M, dtype=np.float32)
        self.l_edge = np.zeros((cfg.N, cfg.M), dtype=np.float32)

        # Generate tasks for the first slot
        self.tasks = self._generate_tasks()

        return self._get_observations()

    # ── Step ───────────────────────────────────────────────────────────────

    def step(
        self, actions: List[int]
    ) -> Tuple[List[np.ndarray], List[float], bool, dict]:
        """
        Advance one time slot.

        Parameters
        ----------
        actions : list of int, length M
            Action for each UE: 0=local, 1..N = offload to UAV 0..N-1

        Returns
        -------
        next_obs : list of np.ndarray
        rewards  : list of float
        done     : bool
        info     : dict
        """
        cfg = self.cfg
        rewards = []
        costs   = []
        n_tasks = sum(1 for t in self.tasks if t is not None)

        for m in range(cfg.M):
            cost, _ = self._process_task(m, actions[m])
            rewards.append(-cost)
            costs.append(cost)

        # Move UEs, advance clock, generate new tasks
        self._move_ues()
        self.t += 1
        done = (self.t >= cfg.I)
        self.tasks = self._generate_tasks()

        info = {
            "avg_cost":        float(np.mean(costs)),
            "n_tasks_generated": n_tasks,
            "t": self.t,
        }

        return self._get_observations(), rewards, done, info

    # ── Core physics ───────────────────────────────────────────────────────

    def _process_task(self, m: int, action: int) -> Tuple[float, bool]:
        """
        Compute cost for UE m taking `action` at time slot self.t.
        Updates queue state in-place.
        Returns (cost, completed).
        """
        cfg  = self.cfg
        task = self.tasks[m]

        if task is None:
            return 0.0, True          # no task — zero cost

        D_mbits, deadline = task
        D_bits = D_mbits * 1e6        # convert to bits
        X = deadline                  # last valid completion slot
        i = self.t

        if action == 0:
            # ── Local processing ───────────────────────────────────────
            # Eq. 6: l_comp = min(i + δ_comp + γ_comp − 1, X)
            gamma_comp = int(np.ceil(
                (D_bits * cfg.sm) / (cfg.f_device * cfg.delta)
            ))
            delta_comp = max(0.0, self.l_comp[m] - i)   # waiting slots (Eq. 9, simplified)
            delta_comp = int(np.ceil(delta_comp))

            finish     = i + delta_comp + gamma_comp - 1
            l_new      = min(finish, X)
            self.l_comp[m] = l_new

            completed  = (finish <= X)
            delay      = (l_new - i + 1)                # Eq. 3: t_local

            # Eq. 12: e_local = κ · D · s_m · (f_device · Δ)²
            energy = cfg.kappa * D_bits * cfg.sm * (cfg.f_device * cfg.delta) ** 2

        else:
            # ── Offload to UAV (action 1..N → UAV index 0..N-1) ───────
            n    = action - 1
            rate = self._tx_rate(m, n)                   # bits / s  (Eq. 1)

            if rate < 1e3:                               # degenerate channel
                return cfg.penalty, False

            # Transmission (Eq. 7)
            gamma_tran = int(np.ceil(D_bits / (rate * cfg.delta)))
            delta_tran = max(0.0, self.l_tran[m] - i)
            delta_tran = int(np.ceil(delta_tran))

            finish_tran = i + delta_tran + gamma_tran - 1
            l_tran_new  = min(finish_tran, X)
            self.l_tran[m] = l_tran_new

            # Task arrives at edge at slot i_star = finish_tran + 1
            i_star = finish_tran + 1
            if i_star > X:
                return cfg.penalty, False

            # Edge processing — CPU equally shared among active queues (Eq. 8)
            # Active queues at UAV n at time i_star = those whose l_edge >= i_star
            active = int(np.sum(self.l_edge[n] > i_star)) + 1  # +1 for this task
            f_shared   = cfg.f_edge_uav / active
            gamma_edge = int(np.ceil(
                (D_bits * cfg.sm) / (f_shared * cfg.delta)
            ))
            delta_edge = max(0.0, self.l_edge[n, m] - i_star)
            delta_edge = int(np.ceil(delta_edge))

            finish_edge = i_star + delta_edge + gamma_edge - 1
            l_edge_new  = min(finish_edge, X)
            self.l_edge[n, m] = l_edge_new

            completed = (finish_edge <= X)
            # Eq. 4+5: total delay = t_tran + t_edge
            delay  = (l_tran_new - i + 1) + (l_edge_new - i_star + 1)

            # Eq. 13: e_tran = Pup · D / (r · Δ)
            energy = (cfg.Pup_uav * D_bits) / (rate * cfg.delta)

        if not completed:
            return cfg.penalty, False

        # Eq. 14: Ψ = w1·t + w2·e
        cost = cfg.w1 * delay + cfg.w2 * energy
        return float(cost), True

    # ── Channel model ──────────────────────────────────────────────────────

    def _channel_gain(self, m: int, n: int) -> float:
        """h_{m,n}(i) = β0 / (‖q_n − p_m‖² + H²)^(j/2)  — Eq. 1"""
        cfg = self.cfg
        dx  = self.uav_pos[n, 0] - self.ue_pos[m, 0]
        dy  = self.uav_pos[n, 1] - self.ue_pos[m, 1]
        return cfg.beta0 / ((dx**2 + dy**2 + cfg.H**2) ** (cfg.j / 2))

    def _tx_rate(self, m: int, n: int) -> float:
        """r_{m,n}(i) = B · log2(1 + P_up · h / σ²)  (bits/s)  — Eq. 1"""
        cfg = self.cfg
        snr = cfg.Pup_uav * self._channel_gain(m, n) / cfg.sigma2
        return cfg.B * np.log2(1.0 + snr)

    # ── Observations ───────────────────────────────────────────────────────

    def _get_observations(self) -> List[np.ndarray]:
        """
        Build flat observation vector for each UE.
        All values normalised to [0, 1].
        """
        cfg = self.cfg
        obs_list = []

        for m in range(cfg.M):
            # (1) UE position — normalised
            ue_xy  = self.ue_pos[m] / np.array([cfg.L, cfg.W], dtype=np.float32)

            # (2) UAV horizontal positions — normalised
            uav_xy = (self.uav_pos / np.array([cfg.L, cfg.W], dtype=np.float32)).flatten()

            # (3) Task size (0 if no task)
            if self.tasks[m] is not None:
                task_feat = np.array([self.tasks[m][0] / cfg.D_max], dtype=np.float32)
            else:
                task_feat = np.array([0.0], dtype=np.float32)

            # (4) Local queue wait — normalised by tau
            local_wait = np.array(
                [max(0.0, self.l_comp[m] - self.t) / cfg.tau], dtype=np.float32
            )

            # (5) Transmission queue wait — normalised by tau
            tran_wait = np.array(
                [max(0.0, self.l_tran[m] - self.t) / cfg.tau], dtype=np.float32
            )

            # (6) This UE's queue residual at each UAV — normalised by tau
            edge_q = np.array(
                [max(0.0, self.l_edge[n, m] - self.t) / cfg.tau for n in range(cfg.N)],
                dtype=np.float32
            )

            # NOTE: global congestion signal (fraction of UEs queued per UAV)
            # intentionally excluded from raw obs.  IL agents are blind to it.
            # GNN-IL agents recover equivalent coordination signal through
            # message passing — that is the contribution this code evaluates.

            obs = np.concatenate([ue_xy, uav_xy, task_feat,
                                   local_wait, tran_wait, edge_q])
            obs_list.append(obs)

        return obs_list

    # ── Graph data for GNN layer ────────────────────────────────────────────

    def get_graph_data(self) -> dict:
        """
        Build the bipartite UE ↔ UAV graph for the GNN layer.

        UE  node features : raw obs vector  (obs_dim,)  — local info only
        UAV node features : [norm_x, norm_y, congestion_level]  (3,)
            congestion = fraction of UEs with active queue at this UAV.
            This is the key signal IL agents lack; the GNN propagates it
            back to UE nodes via Layer-2 (UAV → UE) message passing.

        Edge weight : normalised channel gain h_{m,n}
        """
        cfg = self.cfg
        obs = self._get_observations()

        # ── UE node features ───────────────────────────────────────────
        ue_x = np.stack(obs, axis=0)                        # (M, obs_dim)

        # ── UAV node features ──────────────────────────────────────────
        uav_feats = []
        max_gain  = cfg.beta0 / (cfg.H ** cfg.j)
        for n in range(cfg.N):
            pos_norm   = self.uav_pos[n] / np.array([cfg.L, cfg.W])
            congestion = np.sum(self.l_edge[n] > self.t) / cfg.M
            uav_feats.append(np.array([pos_norm[0], pos_norm[1], congestion],
                                       dtype=np.float32))
        uav_x = np.stack(uav_feats, axis=0)                 # (N, 3)

        # ── UE ↔ UAV edges (fully bipartite) ──────────────────────────
        ue_uav_src, ue_uav_dst, ue_uav_w = [], [], []

        for m in range(cfg.M):
            for n in range(cfg.N):
                gain      = self._channel_gain(m, n)
                norm_gain = float(gain / max_gain)
                ue_uav_src.append(m)
                ue_uav_dst.append(n)
                ue_uav_w.append(norm_gain)

        return {
            "ue_x":  ue_x,                                   # (M, obs_dim)
            "uav_x": uav_x,                                  # (N, 3)
            "ue_uav": (ue_uav_src, ue_uav_dst, ue_uav_w),   # M*N edges
            "n_ues":  cfg.M,
            "n_uavs": cfg.N,
            "obs_dim": self.obs_dim,
        }

    # ── Mobility ───────────────────────────────────────────────────────────

    def _move_ues(self):
        """Random-direction walk; reflect at area boundaries."""
        cfg = self.cfg
        # 30% chance of changing direction each slot
        change = np.random.random(cfg.M) < 0.3
        self.ue_angles[change] = np.random.uniform(0, 2 * np.pi, change.sum())

        dx = cfg.v * cfg.tmov * np.cos(self.ue_angles)
        dy = cfg.v * cfg.tmov * np.sin(self.ue_angles)

        new_x = self.ue_pos[:, 0] + dx
        new_y = self.ue_pos[:, 1] + dy

        # Reflect off walls
        over_x  = new_x > cfg.L;  new_x[over_x]  = 2 * cfg.L - new_x[over_x];  self.ue_angles[over_x]  = np.pi - self.ue_angles[over_x]
        under_x = new_x < 0;      new_x[under_x] = -new_x[under_x];             self.ue_angles[under_x] = np.pi - self.ue_angles[under_x]
        over_y  = new_y > cfg.W;  new_y[over_y]  = 2 * cfg.W - new_y[over_y];  self.ue_angles[over_y]  = -self.ue_angles[over_y]
        under_y = new_y < 0;      new_y[under_y] = -new_y[under_y];             self.ue_angles[under_y] = -self.ue_angles[under_y]

        self.ue_pos[:, 0] = np.clip(new_x, 0, cfg.L)
        self.ue_pos[:, 1] = np.clip(new_y, 0, cfg.W)

    # ── Task generation ────────────────────────────────────────────────────

    def _generate_tasks(self):
        """Each UE independently generates a task with probability P_task."""
        cfg   = self.cfg
        tasks = []
        for m in range(cfg.M):
            if np.random.random() < cfg.P_task:
                D        = float(np.random.uniform(cfg.D_min, cfg.D_max))
                deadline = self.t + cfg.tau - 1
                tasks.append((D, deadline))
            else:
                tasks.append(None)
        return tasks


# ─── Sanity check ──────────────────────────────────────────────────────────

if __name__ == "__main__":
    np.random.seed(42)
    env = NTNMECEnv()

    print(f"Obs dim   : {env.obs_dim}")
    print(f"N actions : {env.n_actions}")
    print(f"N agents  : {env.n_agents}")

    obs = env.reset()
    print(f"\nObs[0] shape : {obs[0].shape}")
    print(f"Obs[0]       : {np.round(obs[0], 3)}")

    # Random policy rollout
    total_cost = 0.0
    for step in range(env.cfg.I):
        actions = [np.random.randint(env.n_actions) for _ in range(env.n_agents)]
        obs, rewards, done, info = env.step(actions)
        total_cost += info["avg_cost"]
        if done:
            break

    print(f"\nRandom policy — mean cost per slot: {total_cost / env.cfg.I:.4f}")

    # Check graph data shapes
    graph = env.get_graph_data()
    print(f"\nGraph: ue_x={graph['ue_x'].shape}, uav_x={graph['uav_x'].shape}")
    print(f"       UE–UAV edges : {len(graph['ue_uav'][0])}")


Writing env.py


In [4]:
%%writefile gnnAgent.py

"""
GNN-IL Agent — shared bipartite GNN encoder + shared Dueling DQN.

Architecture
------------
Graph:  UE nodes ↔ UAV nodes, fully bipartite, edge weight = normalised
        channel gain h_{m,n}.  No UE-UE edges (physically unmotivated).

GNN (2-layer, pure torch — no PyG dependency):

  Layer 1  UE → UAV
    For each UAV n, aggregate UE node features weighted by softmax-
    normalised channel gains → UAV hidden state h_uav  (B, N, gnn_hidden)

  Layer 2  UAV → UE
    For each UE m, aggregate h_uav weighted by softmax-normalised channel
    gains → enriched UE embedding h_ue  (B, M, gnn_out)

DQN:
    input = cat(raw_obs_m, h_ue_m)   dim = obs_dim + gnn_out
    → shared Dueling DQN head (same weights for every UE)

Replay buffer:
    Stores FULL SYSTEM transitions so that the GNN can be re-run inside
    train_step(), keeping the computation graph intact for backward().

    Entry: (ue_obs, uav_feats, edge_w,
            actions, rewards,
            next_ue_obs, next_uav_feats, next_edge_w,
            done)
"""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random
from typing import List, Optional, Tuple

from agent import DuelingDQN       # reuse the same dueling head


# ─── Bipartite GNN Encoder ─────────────────────────────────────────────────

class BipartiteGNNEncoder(nn.Module):
    """
    Two-layer message passing on a UE ↔ UAV bipartite graph.

    Parameters
    ----------
    ue_dim    : dimension of UE node features  (= obs_dim)
    uav_dim   : dimension of UAV node features (= 3)
    hidden    : hidden size after Layer 1
    out_dim   : output embedding size per UE after Layer 2
    """

    def __init__(
        self,
        ue_dim:  int,
        uav_dim: int,
        hidden:  int = 64,
        out_dim: int = 32,
    ):
        super().__init__()

        # Layer 1: [uav_own_feat || weighted_mean(UE_feats)] → hidden
        self.layer1 = nn.Sequential(
            nn.Linear(uav_dim + ue_dim, hidden),
            nn.ReLU(),
        )

        # Layer 2: [ue_own_feat || weighted_mean(h_uav)] → out_dim
        self.layer2 = nn.Sequential(
            nn.Linear(ue_dim + hidden, out_dim),
            nn.ReLU(),
        )

    def forward(
        self,
        ue_x:   torch.Tensor,   # (B, M, ue_dim)
        uav_x:  torch.Tensor,   # (B, N, uav_dim)
        edge_w: torch.Tensor,   # (B, M, N)  raw channel gains
    ) -> torch.Tensor:          # (B, M, out_dim)

        # ── Layer 1: UE → UAV ─────────────────────────────────────────
        # Normalise weights column-wise (over UEs) so each UAV's
        # attention coefficients sum to 1.
        attn1     = F.softmax(edge_w, dim=1)                    # (B, M, N)
        # Weighted mean of UE features arriving at each UAV
        h_agg_uav = torch.bmm(attn1.transpose(1, 2), ue_x)    # (B, N, ue_dim)
        # Concat UAV's own features and project
        h_uav     = self.layer1(
            torch.cat([uav_x, h_agg_uav], dim=-1)              # (B, N, uav_dim+ue_dim)
        )                                                        # (B, N, hidden)

        # ── Layer 2: UAV → UE ─────────────────────────────────────────
        # Normalise weights row-wise (over UAVs) so each UE's
        # attention coefficients sum to 1.
        attn2     = F.softmax(edge_w, dim=2)                    # (B, M, N)
        # Weighted mean of UAV hidden states arriving at each UE
        h_agg_ue  = torch.bmm(attn2, h_uav)                   # (B, M, hidden)
        # Concat UE's own features and project
        h_ue      = self.layer2(
            torch.cat([ue_x, h_agg_ue], dim=-1)               # (B, M, ue_dim+hidden)
        )                                                        # (B, M, out_dim)

        return h_ue


# ─── Global Replay Buffer ──────────────────────────────────────────────────

class GlobalReplayBuffer:
    """
    Stores full-system transitions.  Each entry contains observations,
    graph topology, actions, rewards, and next-step equivalents for
    ALL agents simultaneously.

    This is required so that train_step() can re-run the GNN on the
    stored raw observations and keep the computation graph alive for
    loss.backward() to reach GNN weights.
    """

    def __init__(self, capacity: int = 5_000):
        self.buf = deque(maxlen=capacity)

    def push(
        self,
        ue_obs:        np.ndarray,   # (M, obs_dim)
        uav_feats:     np.ndarray,   # (N, 3)
        edge_w:        np.ndarray,   # (M, N)
        actions:       np.ndarray,   # (M,)  int
        rewards:       np.ndarray,   # (M,)  float
        next_ue_obs:   np.ndarray,   # (M, obs_dim)
        next_uav_feats:np.ndarray,   # (N, 3)
        next_edge_w:   np.ndarray,   # (M, N)
        done:          float,
        task_mask:     np.ndarray,   # (M,)  bool — True if agent had a task
    ):
        self.buf.append((
            ue_obs.astype(np.float32),
            uav_feats.astype(np.float32),
            edge_w.astype(np.float32),
            actions.astype(np.int64),
            rewards.astype(np.float32),
            next_ue_obs.astype(np.float32),
            next_uav_feats.astype(np.float32),
            next_edge_w.astype(np.float32),
            np.float32(done),
            task_mask.astype(bool),
        ))

    def sample(self, batch_size: int) -> Tuple[torch.Tensor, ...]:
        batch = random.sample(self.buf, batch_size)
        (ue_obs, uav_feats, edge_w, actions, rewards,
         next_ue_obs, next_uav_feats, next_edge_w, dones,
         task_masks) = zip(*batch)

        return (
            torch.tensor(np.stack(ue_obs),         dtype=torch.float32),   # (B,M,obs)
            torch.tensor(np.stack(uav_feats),       dtype=torch.float32),   # (B,N,3)
            torch.tensor(np.stack(edge_w),          dtype=torch.float32),   # (B,M,N)
            torch.tensor(np.stack(actions),         dtype=torch.long),      # (B,M)
            torch.tensor(np.stack(rewards),         dtype=torch.float32),   # (B,M)
            torch.tensor(np.stack(next_ue_obs),     dtype=torch.float32),   # (B,M,obs)
            torch.tensor(np.stack(next_uav_feats),  dtype=torch.float32),   # (B,N,3)
            torch.tensor(np.stack(next_edge_w),     dtype=torch.float32),   # (B,M,N)
            torch.tensor(np.array(dones),           dtype=torch.float32),   # (B,)
            torch.tensor(np.stack(task_masks),      dtype=torch.bool),      # (B,M)
        )

    def __len__(self) -> int:
        return len(self.buf)


# ─── GNN-IL Agent ──────────────────────────────────────────────────────────

class GNNILAgent:
    """
    Single shared GNN + DQN serving all M UEs.

    Shared weights mean:
      - All UEs use the same aggregation function (homogeneous agents).
      - Parameter count stays small.
      - One optimizer step updates the policy for the whole fleet.

    Gradient flow (the key fix):
      action selection  →  GNN runs under torch.no_grad()   (fast, no graph)
      train_step()      →  GNN runs WITH grad tracking, then
                           loss.backward() reaches GNN weights correctly.
    """

    def __init__(
        self,
        obs_dim:      int,
        n_uavs:       int,
        n_actions:    int,
        gnn_hidden:   int   = 64,
        gnn_out:      int   = 32,
        dqn_hidden:   int   = 128,
        lr:           float = 1e-3,
        gamma:        float = 0.9,
        batch_size:   int   = 32,
        buffer_cap:   int   = 5_000,
        target_update:int   = 20,
        eps_start:    float = 1.0,
        eps_end:      float = 0.05,
        eps_decay:    float = 0.995,
        device:       str   = "cpu",
    ):
        self.n_actions     = n_actions
        self.gamma         = gamma
        self.batch_size    = batch_size
        self.target_update = target_update
        self.device        = torch.device(device)

        self.eps       = eps_start
        self.eps_end   = eps_end
        self.eps_decay = eps_decay

        self.enriched_dim = obs_dim + gnn_out

        # ── Networks ──────────────────────────────────────────────────
        self.gnn = BipartiteGNNEncoder(
            ue_dim  = obs_dim,
            uav_dim = 3,
            hidden  = gnn_hidden,
            out_dim = gnn_out,
        ).to(self.device)

        # Shared dueling DQN — input is enriched observation
        self.policy_net = DuelingDQN(
            self.enriched_dim, n_actions, dqn_hidden
        ).to(self.device)

        self.target_net = DuelingDQN(
            self.enriched_dim, n_actions, dqn_hidden
        ).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        # Single optimiser covers BOTH gnn and policy_net parameters
        self.optimiser = torch.optim.RMSprop(
            list(self.gnn.parameters()) + list(self.policy_net.parameters()),
            lr=lr,
        )

        self.buffer = GlobalReplayBuffer(buffer_cap)
        self.steps  = 0

    # ── Graph helper ───────────────────────────────────────────────────────

    @staticmethod
    def extract_edge_matrix(graph_data: dict) -> np.ndarray:
        """
        Convert the (src, dst, weight) edge lists from env.get_graph_data()
        into a dense (M, N) numpy matrix.
        Only UE→UAV edges are used; UE-UE edges are ignored entirely.
        """
        M = graph_data["n_ues"]
        N = graph_data["n_uavs"]
        src, dst, w = graph_data["ue_uav"]
        mat = np.zeros((M, N), dtype=np.float32)
        for s, d, wt in zip(src, dst, w):
            mat[s, d] = wt
        return mat                                               # (M, N)

    # ── Action selection ──────────────────────────────────────────────────

    def select_actions(
        self,
        obs_list:   List[np.ndarray],   # length M, each (obs_dim,)
        graph_data: dict,
        greedy:     bool = False,
    ) -> List[int]:
        """
        Forward pass under no_grad → ε-greedy action per UE.
        GNN runs on the full system observation simultaneously.
        """
        with torch.no_grad():
            ue_x   = torch.tensor(
                np.stack(obs_list), dtype=torch.float32
            ).unsqueeze(0).to(self.device)                      # (1, M, obs_dim)

            uav_x  = torch.tensor(
                graph_data["uav_x"], dtype=torch.float32
            ).unsqueeze(0).to(self.device)                      # (1, N, 3)

            edge_w = torch.tensor(
                self.extract_edge_matrix(graph_data), dtype=torch.float32
            ).unsqueeze(0).to(self.device)                      # (1, M, N)

            h_ue     = self.gnn(ue_x, uav_x, edge_w)           # (1, M, gnn_out)
            enriched = torch.cat([ue_x, h_ue], dim=-1)         # (1, M, enriched_dim)
            enriched = enriched.squeeze(0)                      # (M, enriched_dim)

            q_values      = self.policy_net(enriched)           # (M, n_actions)
            greedy_actions = q_values.argmax(dim=1).cpu().numpy()

        if greedy:
            return [int(a) for a in greedy_actions]

        return [
            random.randint(0, self.n_actions - 1)
            if random.random() < self.eps
            else int(greedy_actions[m])
            for m in range(len(obs_list))
        ]

    # ── Store transition ──────────────────────────────────────────────────

    def store(
        self,
        obs_list:        List[np.ndarray],
        graph_data:      dict,
        actions:         List[int],
        rewards:         List[float],
        next_obs_list:   List[np.ndarray],
        next_graph_data: dict,
        done:            bool,
        task_mask:       List[bool],
    ):
        self.buffer.push(
            ue_obs         = np.stack(obs_list),
            uav_feats      = graph_data["uav_x"],
            edge_w         = self.extract_edge_matrix(graph_data),
            actions        = np.array(actions),
            rewards        = np.array(rewards),
            next_ue_obs    = np.stack(next_obs_list),
            next_uav_feats = next_graph_data["uav_x"],
            next_edge_w    = self.extract_edge_matrix(next_graph_data),
            done           = float(done),
            task_mask      = np.array(task_mask),
        )

    # ── Training step ─────────────────────────────────────────────────────

    def train_step(self) -> Optional[float]:
        """
        Sample one global mini-batch and update GNN + policy_net jointly.

        The GNN is called INSIDE this function so that the full computation
        graph — GNN encoder → DQN head → loss — exists when backward() runs.
        Both GNN and DQN weights receive gradients.
        """
        if len(self.buffer) < self.batch_size:
            return None

        (ue_obs, uav_feats, edge_w,
         actions, rewards,
         next_ue_obs, next_uav_feats, next_edge_w,
         dones, task_masks) = [t.to(self.device) for t in self.buffer.sample(self.batch_size)]

        # Shapes:
        #   ue_obs, next_ue_obs  : (B, M, obs_dim)
        #   uav_feats            : (B, N, 3)
        #   edge_w               : (B, M, N)
        #   actions, rewards     : (B, M)
        #   dones                : (B,)
        #   task_masks           : (B, M)  bool

        B, M, _ = ue_obs.shape

        # ── Current Q values ──────────────────────────────────────────
        h_ue     = self.gnn(ue_obs, uav_feats, edge_w)         # (B, M, gnn_out)
        enriched = torch.cat([ue_obs, h_ue], dim=-1)           # (B, M, enriched_dim)
        enriched_flat  = enriched.view(B * M, -1)              # (B*M, enriched_dim)
        q_all          = self.policy_net(enriched_flat)         # (B*M, n_actions)
        actions_flat   = actions.view(B * M)                   # (B*M,)
        q_pred = q_all.gather(
            1, actions_flat.unsqueeze(1)
        ).squeeze(1)                                            # (B*M,)

        # ── Target Q values (Double DQN, no grad) ────────────────────
        with torch.no_grad():
            h_ue_next      = self.gnn(next_ue_obs, next_uav_feats, next_edge_w)
            enriched_next  = torch.cat([next_ue_obs, h_ue_next], dim=-1)
            enriched_nflat = enriched_next.view(B * M, -1)    # (B*M, enriched_dim)

            next_actions   = self.policy_net(enriched_nflat).argmax(dim=1)
            next_q         = self.target_net(enriched_nflat).gather(
                1, next_actions.unsqueeze(1)
            ).squeeze(1)                                        # (B*M,)

            dones_exp    = dones.unsqueeze(1).expand(B, M).reshape(B * M)
            rewards_flat = rewards.view(B * M)
            q_target     = rewards_flat + self.gamma * next_q * (1.0 - dones_exp)

        # ── Loss: only over agents that had a task this slot ──────────
        # task_masks flat: (B*M,) bool
        mask_flat = task_masks.view(B * M)                     # (B*M,) bool
        if mask_flat.sum() == 0:
            return None                                         # nothing to learn from

        loss = F.smooth_l1_loss(q_pred[mask_flat], q_target[mask_flat])

        self.optimiser.zero_grad()
        loss.backward()                # gradients flow through DQN AND GNN
        nn.utils.clip_grad_norm_(
            list(self.gnn.parameters()) + list(self.policy_net.parameters()),
            max_norm=10.0,
        )
        self.optimiser.step()

        self.steps += 1
        if self.steps % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return float(loss.item())

    # ── ε schedule ────────────────────────────────────────────────────────

    def decay_epsilon(self):
        """Call once per episode."""
        self.eps = max(self.eps_end, self.eps * self.eps_decay)

    # ── Persistence ───────────────────────────────────────────────────────

    def save(self, path: str):
        torch.save({
            "gnn":        self.gnn.state_dict(),
            "policy_net": self.policy_net.state_dict(),
        }, path)

    def load(self, path: str):
        ckpt = torch.load(path, map_location=self.device)
        self.gnn.load_state_dict(ckpt["gnn"])
        self.policy_net.load_state_dict(ckpt["policy_net"])
        self.target_net.load_state_dict(ckpt["policy_net"])

Writing gnnAgent.py


In [5]:
%%writefile train.py

"""
IL Training Loop.

Runs M independent dueling-DQN agents, each training on its own
local replay buffer — exactly the IL-based framework from the paper.

Usage:
    python train_il.py                        # default config
    python train_il.py --n_ues 10 --n_uavs 2 --episodes 500
"""

import argparse
import numpy as np
import torch
import random
import json
import os
from typing import List

from env   import NTNMECEnv, EnvConfig
from agent import ILAgent


# ─── Helpers ───────────────────────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_agents(env: NTNMECEnv, device: str, **agent_kwargs) -> List[ILAgent]:
    return [
        ILAgent(
            obs_dim   = env.obs_dim,
            n_actions = env.n_actions,
            device    = device,
            **agent_kwargs,
        )
        for _ in range(env.n_agents)
    ]


# ─── One episode ───────────────────────────────────────────────────────────

def run_episode(
    env:     NTNMECEnv,
    agents:  List[ILAgent],
    train:   bool = True,
) -> dict:
    """
    Roll out one full episode.

    Returns
    -------
    dict with:
        avg_cost      — mean cost across all UEs and time slots
        total_loss    — mean training loss (nan if train=False)
        eps           — current epsilon of agent 0
    """
    obs_list  = env.reset()
    ep_costs  = []
    ep_losses = []

    for _ in range(env.cfg.I):
        # Capture which agents have a task THIS slot, before step() advances time
        had_task = [t is not None for t in env.tasks]

        actions = [
            agent.select_action(obs, greedy=not train)
            for agent, obs in zip(agents, obs_list)
        ]

        next_obs_list, rewards, done, info = env.step(actions)

        if train:
            for m, agent in enumerate(agents):
                if not had_task[m]:
                    # No task this slot — action had no effect, skip storing.
                    # Avoids polluting the buffer with zero-reward noise and
                    # prevents agents from learning that offloading is "risky"
                    # on slots where nothing actually happens.
                    continue
                agent.store(
                    obs_list[m], actions[m], rewards[m],
                    next_obs_list[m], float(done)
                )
                loss = agent.train_step()
                if loss is not None:
                    ep_losses.append(loss)

        ep_costs.append(info["avg_cost"])
        obs_list = next_obs_list

        if done:
            break

    return {
        "avg_cost":   float(np.mean(ep_costs)),
        "total_loss": float(np.mean(ep_losses)) if ep_losses else float("nan"),
        "eps":        agents[0].eps,
    }


# ─── Full training run ─────────────────────────────────────────────────────

def train(args) -> dict:
    set_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() and not args.cpu else "cpu"
    print(f"Device: {device}")

    cfg = EnvConfig(
        M      = args.n_ues,
        N      = args.n_uavs,
        I      = args.steps_per_ep,
        P_task = args.task_prob,
    )
    env    = NTNMECEnv(cfg)
    agents = make_agents(
        env,
        device        = device,
        hidden        = args.hidden,
        lr            = args.lr,
        gamma         = args.gamma,
        batch_size    = args.batch_size,
        target_update = args.target_update,
        eps_decay     = args.eps_decay,
    )

    history   = []
    best_cost = float("inf")

    print(f"\nTraining IL — {args.n_ues} UEs, {args.n_uavs} UAVs, "
          f"{args.episodes} episodes\n")
    print(f"{'Episode':>8}  {'AvgCost':>10}  {'Loss':>10}  {'Eps':>6}")
    print("-" * 42)

    for ep in range(1, args.episodes + 1):
        stats = run_episode(env, agents, train=True)
        history.append(stats)

        if ep % args.log_every == 0:
            print(f"{ep:>8}  {stats['avg_cost']:>10.4f}  "
                  f"{stats['total_loss']:>10.4f}  {stats['eps']:>6.3f}")

        # Decay ε once per episode for all agents
        for agent in agents:
            agent.decay_epsilon()

        if stats["avg_cost"] < best_cost:
            best_cost = stats["avg_cost"]
            if args.save_dir:
                _save_agents(agents, args.save_dir, tag="il_best")

    # Final greedy eval
    eval_costs = []
    for _ in range(args.eval_episodes):
        s = run_episode(env, agents, train=False)
        eval_costs.append(s["avg_cost"])

    eval_mean = float(np.mean(eval_costs))
    eval_std  = float(np.std(eval_costs))
    print(f"\nEval ({args.eval_episodes} eps): "
          f"mean={eval_mean:.4f}  std={eval_std:.4f}")

    results = {
        "method":     "IL",
        "config":     vars(args),
        "history":    history,
        "eval_mean":  eval_mean,
        "eval_std":   eval_std,
        "best_train": best_cost,
    }

    if args.save_dir:
        os.makedirs(args.save_dir, exist_ok=True)
        _save_agents(agents, args.save_dir, tag="il_final")
        with open(os.path.join(args.save_dir, "il_results.json"), "w") as f:
            json.dump(results, f, indent=2)
        print(f"Saved to {args.save_dir}/")

    return results


# ─── Persistence ───────────────────────────────────────────────────────────

def _save_agents(agents: List[ILAgent], save_dir: str, tag: str):
    os.makedirs(save_dir, exist_ok=True)
    for m, agent in enumerate(agents):
        path = os.path.join(save_dir, f"{tag}_agent{m}.pt")
        torch.save(agent.policy_net.state_dict(), path)


def load_agents(
    env:      NTNMECEnv,
    save_dir: str,
    tag:      str,
    device:   str = "cpu",
) -> List[ILAgent]:
    agents = make_agents(env, device=device)
    for m, agent in enumerate(agents):
        path = os.path.join(save_dir, f"{tag}_agent{m}.pt")
        agent.policy_net.load_state_dict(torch.load(path, map_location=device))
        agent.policy_net.eval()
    return agents


# ─── Baselines ─────────────────────────────────────────────────────────────

def run_random_policy(env: NTNMECEnv, n_episodes: int = 20) -> float:
    costs = []
    for _ in range(n_episodes):
        env.reset()
        ep_costs = []
        for _ in range(env.cfg.I):
            actions = [random.randint(0, env.n_actions - 1)
                       for _ in range(env.n_agents)]
            _, _, done, info = env.step(actions)
            ep_costs.append(info["avg_cost"])
            if done:
                break
        costs.append(np.mean(ep_costs))
    return float(np.mean(costs))


def run_local_policy(env: NTNMECEnv, n_episodes: int = 20) -> float:
    costs = []
    for _ in range(n_episodes):
        env.reset()
        ep_costs = []
        for _ in range(env.cfg.I):
            actions = [0] * env.n_agents
            _, _, done, info = env.step(actions)
            ep_costs.append(info["avg_cost"])
            if done:
                break
        costs.append(np.mean(ep_costs))
    return float(np.mean(costs))


# ─── CLI ───────────────────────────────────────────────────────────────────

def get_args():
    p = argparse.ArgumentParser()

    # Environment
    p.add_argument("--n_ues",        type=int,   default=10)
    p.add_argument("--n_uavs",       type=int,   default=2)
    p.add_argument("--steps_per_ep", type=int,   default=100)
    p.add_argument("--task_prob",    type=float, default=0.3)

    # Training
    p.add_argument("--episodes",      type=int,   default=500)
    p.add_argument("--eval_episodes", type=int,   default=20)
    p.add_argument("--hidden",        type=int,   default=128)
    p.add_argument("--lr",            type=float, default=1e-3)
    p.add_argument("--gamma",         type=float, default=0.9)
    p.add_argument("--batch_size",    type=int,   default=32)
    p.add_argument("--target_update", type=int,   default=20)
    p.add_argument("--eps_decay",     type=float, default=0.995)

    # Misc
    p.add_argument("--seed",      type=int,  default=42)
    p.add_argument("--log_every", type=int,  default=50)
    p.add_argument("--save_dir",  type=str,  default="checkpoints")
    p.add_argument("--cpu",       action="store_true")

    return p.parse_args()


if __name__ == "__main__":
    args = get_args()
    results = train(args)

    # Print naive baselines alongside
    cfg = EnvConfig(M=args.n_ues, N=args.n_uavs)
    env = NTNMECEnv(cfg)
    set_seed(args.seed)

    rnd = run_random_policy(env)
    loc = run_local_policy(env)
    print(f"\nBaselines  —  random: {rnd:.4f}  |  local: {loc:.4f}")
    print(f"IL eval    —  {results['eval_mean']:.4f} ± {results['eval_std']:.4f}")

Writing train.py


In [6]:
%%writefile trainGnn.py

"""
GNN-IL Training Loop.

Trains a single shared BipartiteGNNEncoder + DuelingDQN over all UEs.
Structure mirrors train_il.py exactly for a fair comparison.

Usage:
    python train_gnn_il.py                        # default config
    python train_gnn_il.py --n_ues 10 --n_uavs 2 --episodes 500
"""

import argparse
import numpy as np
import torch
import random
import json
import os

from env       import NTNMECEnv, EnvConfig
from gnnAgent import GNNILAgent


# ─── Helpers ───────────────────────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ─── One episode ───────────────────────────────────────────────────────────

def run_episode(
    env:   NTNMECEnv,
    agent: GNNILAgent,
    train: bool = True,
) -> dict:
    """
    Roll out one full episode.

    At each timestep we need the graph snapshot TWICE:
      - Current  graph → action selection + storing transition
      - Next     graph → stored in transition for train_step()
    Both are fetched from env.get_graph_data() at the right moment.
    """
    obs_list   = env.reset()
    graph_data = env.get_graph_data()          # graph at t=0

    ep_costs  = []
    ep_losses = []

    for _ in range(env.cfg.I):
        # Capture which agents have a task THIS slot, before step() advances time
        had_task = [t is not None for t in env.tasks]

        actions = agent.select_actions(obs_list, graph_data, greedy=not train)

        next_obs_list, rewards, done, info = env.step(actions)
        next_graph_data = env.get_graph_data()  # graph at t+1

        if train:
            # Only store if at least one UE had a task this slot.
            # task_mask passed into the buffer so train_step() restricts
            # the loss to agents that actually had decisions to make.
            if any(had_task):
                agent.store(
                    obs_list, graph_data,
                    actions, rewards,
                    next_obs_list, next_graph_data,
                    done,
                    task_mask=had_task,
                )
            loss = agent.train_step()
            if loss is not None:
                ep_losses.append(loss)

        ep_costs.append(info["avg_cost"])
        obs_list   = next_obs_list
        graph_data = next_graph_data

        if done:
            break

    return {
        "avg_cost":   float(np.mean(ep_costs)),
        "total_loss": float(np.mean(ep_losses)) if ep_losses else float("nan"),
        "eps":        agent.eps,
    }


# ─── Full training run ─────────────────────────────────────────────────────

def train(args) -> dict:
    set_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() and not args.cpu else "cpu"
    print(f"Device: {device}")

    cfg = EnvConfig(
        M      = args.n_ues,
        N      = args.n_uavs,
        I      = args.steps_per_ep,
        P_task = args.task_prob,
    )
    env = NTNMECEnv(cfg)

    agent = GNNILAgent(
        obs_dim       = env.obs_dim,
        n_uavs        = cfg.N,
        n_actions     = env.n_actions,
        gnn_hidden    = args.gnn_hidden,
        gnn_out       = args.gnn_out,
        dqn_hidden    = args.dqn_hidden,
        lr            = args.lr,
        gamma         = args.gamma,
        batch_size    = args.batch_size,
        buffer_cap    = args.buffer_cap,
        target_update = args.target_update,
        eps_decay     = args.eps_decay,
        device        = device,
    )

    history   = []
    best_cost = float("inf")

    print(f"\nTraining GNN-IL — {args.n_ues} UEs, {args.n_uavs} UAVs, "
          f"{args.episodes} episodes")
    print(f"GNN: hidden={args.gnn_hidden}, out={args.gnn_out}  "
          f"| enriched_dim={env.obs_dim + args.gnn_out}\n")
    print(f"{'Episode':>8}  {'AvgCost':>10}  {'Loss':>10}  {'Eps':>6}")
    print("-" * 42)

    for ep in range(1, args.episodes + 1):
        stats = run_episode(env, agent, train=True)
        history.append(stats)

        agent.decay_epsilon()

        if ep % args.log_every == 0:
            print(f"{ep:>8}  {stats['avg_cost']:>10.4f}  "
                  f"{stats['total_loss']:>10.4f}  {stats['eps']:>6.3f}")

        if stats["avg_cost"] < best_cost:
            best_cost = stats["avg_cost"]
            if args.save_dir:
                os.makedirs(args.save_dir, exist_ok=True)
                agent.save(os.path.join(args.save_dir, "gnn_il_best.pt"))

    # Final greedy eval — identical to train_il.py eval
    eval_costs = []
    for _ in range(args.eval_episodes):
        s = run_episode(env, agent, train=False)
        eval_costs.append(s["avg_cost"])

    eval_mean = float(np.mean(eval_costs))
    eval_std  = float(np.std(eval_costs))
    print(f"\nEval ({args.eval_episodes} eps): "
          f"mean={eval_mean:.4f}  std={eval_std:.4f}")

    results = {
        "method":     "GNN-IL",
        "config":     vars(args),
        "history":    history,
        "eval_mean":  eval_mean,
        "eval_std":   eval_std,
        "best_train": best_cost,
    }

    if args.save_dir:
        os.makedirs(args.save_dir, exist_ok=True)
        agent.save(os.path.join(args.save_dir, "gnn_il_final.pt"))
        with open(os.path.join(args.save_dir, "gnn_il_results.json"), "w") as f:
            json.dump(results, f, indent=2)
        print(f"Saved to {args.save_dir}/")

    return results


# ─── CLI ───────────────────────────────────────────────────────────────────

def get_args():
    p = argparse.ArgumentParser()

    # Environment — keep defaults identical to train_il.py for fair comparison
    p.add_argument("--n_ues",        type=int,   default=10)
    p.add_argument("--n_uavs",       type=int,   default=2)
    p.add_argument("--steps_per_ep", type=int,   default=100)
    p.add_argument("--task_prob",    type=float, default=0.3)

    # GNN
    p.add_argument("--gnn_hidden",   type=int,   default=64)
    p.add_argument("--gnn_out",      type=int,   default=32)

    # DQN
    p.add_argument("--dqn_hidden",   type=int,   default=128)

    # Training — identical defaults to train_il.py
    p.add_argument("--episodes",      type=int,   default=500)
    p.add_argument("--eval_episodes", type=int,   default=20)
    p.add_argument("--lr",            type=float, default=1e-3)
    p.add_argument("--gamma",         type=float, default=0.9)
    p.add_argument("--batch_size",    type=int,   default=32)
    p.add_argument("--buffer_cap",    type=int,   default=5_000)
    p.add_argument("--target_update", type=int,   default=20)
    p.add_argument("--eps_decay",     type=float, default=0.995)

    # Misc
    p.add_argument("--seed",      type=int,  default=42)
    p.add_argument("--log_every", type=int,  default=50)
    p.add_argument("--save_dir",  type=str,  default="checkpoints")
    p.add_argument("--cpu",       action="store_true")

    return p.parse_args()


if __name__ == "__main__":
    args    = get_args()
    results = train(args)
    print(f"\nGNN-IL eval  —  {results['eval_mean']:.4f} ± {results['eval_std']:.4f}")

Writing trainGnn.py


In [ ]:
import os
import subprocess
from itertools import product

ues_list = [5, 10, 20, 30]
seeds_list = [42, 52, 62, 72, 82] # 5 seeds
episodes = 500

BASE_DIR = '/content/drive/MyDrive/NTN_MEC_Ablation'

total_runs = len(ues_list) * len(seeds_list) * 2 # *2 for Standard and GNN
current_run = 0

for ues, seed in product(ues_list, seeds_list):
    print(f"\n{'='*50}")
    print(f"UEs={ues} | SEED={seed}")
    print(f"{'='*50}")

    # --- 1. Standard IL ---
    current_run += 1
    std_dir = os.path.join(BASE_DIR, f"ue{ues}", f"seed{seed}", "standard")

    if not os.path.exists(os.path.join(std_dir, "il_results.json")):
        print(f"[{current_run}/{total_runs}] Training Standard IL...")
        cmd_std = [
            "python", "train.py",
            "--episodes", str(episodes),
            "--log_every", "100", # Log more frequently in Colab to ensure it's alive
            "--n_ues", str(ues),
            "--seed", str(seed),
            "--save_dir", std_dir
        ]
        subprocess.run(cmd_std, check=True)
    else:
        print(f"[{current_run}/{total_runs}] Skipping Standard IL (Already exists in Drive)")

    # --- 2. GNN IL ---
    current_run += 1
    gnn_dir = os.path.join(BASE_DIR, f"ue{ues}", f"seed{seed}", "gnn")

    if not os.path.exists(os.path.join(gnn_dir, "gnn_il_results.json")):
        print(f"[{current_run}/{total_runs}] Training GNN IL...")
        cmd_gnn = [
            "python", "trainGnn.py",
            "--episodes", str(episodes),
            "--log_every", "100",
            "--n_ues", str(ues),
            "--seed", str(seed),
            "--save_dir", gnn_dir
        ]
        subprocess.run(cmd_gnn, check=True)
    else:
        print(f"[{current_run}/{total_runs}] Skipping GNN IL (Already exists in Drive)")

print("\n All multi-seed ablation experiments completed and saved to Google Drive!")


UEs=5 | SEED=42
[1/40] Skipping Standard IL (Already exists in Drive)
[2/40] Skipping GNN IL (Already exists in Drive)

UEs=5 | SEED=52
[3/40] Skipping Standard IL (Already exists in Drive)
[4/40] Skipping GNN IL (Already exists in Drive)

UEs=5 | SEED=62
[5/40] Skipping Standard IL (Already exists in Drive)
[6/40] Skipping GNN IL (Already exists in Drive)

UEs=5 | SEED=72
[7/40] Skipping Standard IL (Already exists in Drive)
[8/40] Skipping GNN IL (Already exists in Drive)

UEs=5 | SEED=82
[9/40] Skipping Standard IL (Already exists in Drive)
[10/40] Skipping GNN IL (Already exists in Drive)

UEs=10 | SEED=42
[11/40] Skipping Standard IL (Already exists in Drive)
[12/40] Skipping GNN IL (Already exists in Drive)

UEs=10 | SEED=52
[13/40] Skipping Standard IL (Already exists in Drive)
[14/40] Skipping GNN IL (Already exists in Drive)

UEs=10 | SEED=62
[15/40] Skipping Standard IL (Already exists in Drive)
[16/40] Skipping GNN IL (Already exists in Drive)

UEs=10 | SEED=72
[17/40] Skip